# WinLOLBIN × VRAVEN DarkSide

Run top → bottom. One analysis per cell.

`Models/winlolbin_mlp/` · benign=`0` · malware=`1`


## 0 — install


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

VRAVEN_ROOT = next(
    (candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "vraven").is_dir()),
    None,
)
if VRAVEN_ROOT is None:
    raise RuntimeError("Start Jupyter from the root of a VRAVEN source checkout")
assert VRAVEN_ROOT.is_dir()

def _missing(name: str) -> bool:
    return importlib.util.find_spec(name) is None

needs = []
if _missing("vraven"):
    needs.append(("vraven", ["-e", str(VRAVEN_ROOT)]))
if _missing("torch"):
    needs.append(("torch", ["torch"]))
if _missing("plotly"):
    needs.append(("plotly", ["plotly"]))
if _missing("sklearn"):
    needs.append(("scikit-learn", ["scikit-learn"]))
if _missing("scipy"):
    needs.append(("scipy", ["scipy"]))

if needs:
    print("Installing:", [n for n, _ in needs])
    for name, args in needs:
        cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
        print("$", " ".join(cmd))
        subprocess.check_call(cmd)
    print("Restart kernel, then continue.")
else:
    print("Kernel OK.")
    print(sys.executable)


## 1 — paths


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import torch

VRAVEN_ROOT = next(
    (candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "vraven").is_dir()),
    None,
)
if VRAVEN_ROOT is None:
    raise RuntimeError("Start Jupyter from the root of a VRAVEN source checkout")
EXAMPLE = VRAVEN_ROOT / "examples" / "windows_malware_mlp"
MODEL_DIR = VRAVEN_ROOT / "Models" / "winlolbin_mlp"
SRC = VRAVEN_ROOT / "src"

for path in (EXAMPLE, MODEL_DIR, SRC):
    assert path.exists(), f"Missing: {path}"

for path in (str(SRC), str(EXAMPLE)):
    if path in sys.path:
        sys.path.remove(path)
    sys.path.insert(0, path)

for name in list(sys.modules):
    if name == "vraven" or name.startswith("vraven."):
        del sys.modules[name]

from features import CLASS_NAMES, CONTEXT, DISPLAY_NAMES, FEATURES, STRUCTURAL, SUSPICIOUS
from model import MalwareMLP

import vraven
from vraven import VRavenTheme
from vraven.reasoning.fingerprint import shared_reasoning_fingerprints  # noqa: F401

VRavenTheme.apply()

TARGET = 1
N_SAMPLES = 32
PLOT_MODE = "interactive_dark"

print("vraven", getattr(vraven, "__version__", "?"))
print("python", sys.executable)
print("model", MODEL_DIR)


## 2 — load model


In [ ]:
payload = torch.load(MODEL_DIR / "model.pt", map_location="cpu", weights_only=True)
arch = payload["architecture"]
feature_names = list(payload["feature_names"])
display_names = [DISPLAY_NAMES.get(n, n) for n in feature_names]

model = MalwareMLP(
    input_dim=int(arch["input_dim"]),
    hidden_sizes=tuple(arch["hidden_sizes"]),
    num_classes=int(arch["num_classes"]),
)
model.load_state_dict(payload["model_state"])
model.eval()
print(arch)


## 3 — load samples


In [ ]:
sample = np.load(MODEL_DIR / "explain_sample.npz")
features_all = sample["features_scaled"].astype(np.float32)
labels_all = sample["labels"].astype(int)
n = min(N_SAMPLES, len(labels_all))

idx0 = np.flatnonzero(labels_all == 0)[: n // 2]
idx1 = np.flatnonzero(labels_all == 1)[: n - len(idx0)]
keep = np.concatenate([idx0, idx1]) if len(idx0) and len(idx1) else np.arange(n)

features = features_all[keep]
labels = labels_all[keep]
label_names = [CLASS_NAMES[int(v)] for v in labels]
x = torch.from_numpy(features)

with torch.no_grad():
    probs = torch.softmax(model(x), dim=-1).cpu().numpy()

print("batch", tuple(x.shape), "benign", int((labels == 0).sum()), "malware", int((labels == 1).sum()))


---
# Analyses


## DS00 — setup


In [ ]:
from vraven.darkside import (
    DarkSideSession,
    InputFeatureGroup,
    TraceDataset,
    UnitGroup,
    analyse_adversarial_rerouting,
    analyse_circuit_multiplicity,
    analyse_cost_evidence_tradeoff,
    analyse_explanation_uncertainty,
    analyse_failure_anatomy,
    analyse_high_confidence_ood,
    analyse_method_disagreement,
    analyse_prediction_depth,
    analyse_representation_drift,
    analyse_stability_geometry,
    analyse_topological_shift,
    audit_unused_capacity,
    build_circuit_wiring,
    compare_class_conditionals,
    discover_decision_circuitry,
    inspect_internal_anatomy,
    layerwise_shortcut_probe,
    map_class_conditional_unit_field,
    print_internal_anatomy,
    recommend_features,
)

focus = int(np.argmax(probs[:, 1]))
x_focus = x[focus : focus + 1].clone()
session = DarkSideSession(model, x_focus, target=TARGET)

LINEAR_LAYERS = ("net.0", "net.2", "net.4", "net.6")

idx = {name: FEATURES.index(name) for name in FEATURES}
lolbin_groups = (
    InputFeatureGroup(tuple(idx[n] for n in SUSPICIOUS[:8]), label="lolbin_flags"),
    InputFeatureGroup(tuple(idx[n] for n in STRUCTURAL[:8]), label="cmd_structure"),
    InputFeatureGroup(tuple(idx[n] for n in CONTEXT[:8]), label="process_context"),
)

_ds_reports = [
    vraven.explain(model, x[i : i + 1], target=TARGET, mode="gradient")
    for i in range(len(labels))
]
dark_dataset = TraceDataset.from_reports(
    _ds_reports,
    true_labels=labels.tolist(),
    sample_ids=[f"{label_names[i]}_{i}" for i in range(len(labels))],
)

noise = 0.35 * x.std()
x_noisy = x + noise * torch.randn_like(x)
_noisy_reports = [
    vraven.explain(model, x_noisy[i : i + 1], target=TARGET, mode="gradient")
    for i in range(len(labels))
]
dark_dataset_noisy = TraceDataset.from_reports(
    _noisy_reports,
    true_labels=labels.tolist(),
    sample_ids=[f"{label_names[i]}_{i}" for i in range(len(labels))],
)

print("focus", focus, label_names[focus], float(probs[focus, 1]))
print("layers", LINEAR_LAYERS)
print("dataset", len(dark_dataset.sample_ids), "samples")
print([f.name for f in recommend_features("malware")])


## DS01 — decision circuitry


In [ ]:
circuit = session.decision_circuitry(
    layer_names=LINEAR_LAYERS[:2],
    group_size=16,
    relative_score_tolerance=0.05,
    maximum_groups=128,
)
print(circuit.summary())
circuit.figure(mode=PLOT_MODE).show()


## DS02 — shortcut detection


In [ ]:
shortcut = session.shortcut_detection(lolbin_groups, baseline="zero")
print(shortcut.summary())
shortcut.figure(mode=PLOT_MODE).show()


## DS03 — class-conditional internals


In [ ]:
class_cond = compare_class_conditionals(
    dark_dataset,
    class_a=0,
    class_b=1,
    label_source="true",
    top_units=25,
)
print(class_cond.summary())
class_cond.figure(mode=PLOT_MODE).show()


## DS04 — representation drift


In [ ]:
drift = analyse_representation_drift(
    dark_dataset,
    dark_dataset_noisy,
    sliced_projections=32,
    seed=17,
)
print(drift.summary())
drift.figure(mode=PLOT_MODE).show()


## DS05 — failure anatomy


In [ ]:
# synthetic confusion cohorts (model is near-perfect on this slice)
pred = np.asarray(dark_dataset.predicted_labels, dtype=object).copy()
true = np.asarray(dark_dataset.true_labels, dtype=object)
benign_i = np.flatnonzero(true == 0)[:3]
malware_i = np.flatnonzero(true == 1)[:3]
pred[benign_i] = 1   # FP
pred[malware_i] = 0  # FN
audit = TraceDataset(
    sample_ids=dark_dataset.sample_ids,
    true_labels=dark_dataset.true_labels,
    predicted_labels=pred,
    target_scores=dark_dataset.target_scores,
    representations=dark_dataset.representations,
    unit_influences=dark_dataset.unit_influences,
    unit_energies=dark_dataset.unit_energies,
    activation_rms=dark_dataset.activation_rms,
    timing_ms=dark_dataset.timing_ms,
    fingerprints=dark_dataset.fingerprints,
    metadata=dict(dark_dataset.metadata),
)
failure = analyse_failure_anatomy(audit, failure="fp", reference="tp", positive_label=1)
print(failure.summary())
failure.figure(mode=PLOT_MODE).show()


## DS06 — stability geometry


In [ ]:
stability = analyse_stability_geometry(
    dark_dataset,
    dark_dataset_noisy,
    collapse_threshold=0.70,
    neighbours=5,
)
print(stability.summary())
stability.figure(mode=PLOT_MODE).show()


## DS07 — method disagreement


In [ ]:
grad_vec = np.abs(
    np.asarray(_ds_reports[0].trace.inputs.attribution_values, dtype=np.float64)
).reshape(-1)
lrp_feature = np.zeros_like(grad_vec)
count = 0
for i in range(min(16, x.shape[0])):
    single = vraven.explain_lrp(model, x[i : i + 1], target=TARGET)
    if single.input_relevance is None:
        continue
    vec = np.abs(np.asarray(single.input_relevance, dtype=np.float64)).reshape(-1)
    if vec.size != grad_vec.size:
        continue
    lrp_feature += vec
    count += 1
assert count > 0
consistency_explanations = {
    "input_times_gradient": grad_vec.tolist(),
    "lrp_epsilon": (lrp_feature / count).tolist(),
}

disagreement = analyse_method_disagreement(
    {
        "input_x_gradient": consistency_explanations["input_times_gradient"],
        "lrp_epsilon": consistency_explanations["lrp_epsilon"],
    },
    top_k=20,
)
print(disagreement.summary())
disagreement.figure(mode=PLOT_MODE).show()


## DS08 — unused capacity


In [ ]:
capacity = audit_unused_capacity(dark_dataset)
print(capacity.summary())
capacity.figure(mode=PLOT_MODE).show()


## DS09 — adversarial / evasion rerouting


In [ ]:
reroute = analyse_adversarial_rerouting(
    dark_dataset,
    dark_dataset_noisy,
    input_l2=(x_noisy - x).reshape(len(labels), -1).pow(2).sum(dim=1).sqrt().cpu().numpy(),
    neighbours=5,
)
print(reroute.summary())
reroute.figure(mode=PLOT_MODE).show()


## DS10 — cost–evidence tradeoff


In [ ]:
cost = analyse_cost_evidence_tradeoff(dark_dataset, evidence="activation_rms")
print(cost.summary())
cost.figure(mode=PLOT_MODE).show()


## DS11 — prediction depth


In [ ]:
# stratified halves so both classes appear in reference and query
idx0 = np.flatnonzero(labels == 0)
idx1 = np.flatnonzero(labels == 1)
ref_idx = np.concatenate([idx0[: len(idx0) // 2], idx1[: len(idx1) // 2]])
qry_idx = np.concatenate([idx0[len(idx0) // 2 :], idx1[len(idx1) // 2 :]])
assert len(ref_idx) >= 2 and len(qry_idx) >= 2

depth = analyse_prediction_depth(
    dark_dataset.subset(ref_idx.tolist()),
    dark_dataset.subset(qry_idx.tolist()),
    neighbours=min(3, len(ref_idx) - 1),
)
print(depth.summary())
depth.figure(mode=PLOT_MODE).show()


## DS12 — layerwise shortcut probe


In [ ]:
probe = layerwise_shortcut_probe(
    dark_dataset,
    shortcut_labels=labels.tolist(),
    folds=3,
    seed=17,
)
print(probe.summary())
probe.figure(mode=PLOT_MODE).show()


## DS13 — circuit multiplicity


In [ ]:
masks = {}
labels_comp = None
for tag, kwargs in {
    "seed_a": {"relative_score_tolerance": 0.05},
    "seed_b": {"relative_score_tolerance": 0.08},
    "mean_baseline": {"baseline": "mean", "relative_score_tolerance": 0.05},
}.items():
    r = discover_decision_circuitry(
        model,
        x_focus,
        target=TARGET,
        layer_names=LINEAR_LAYERS[:2],
        group_size=16,
        maximum_groups=128,
        **kwargs,
    )
    retained = {row["group"] for row in r.tables["groups"] if row.get("retained")}
    all_groups = [row["group"] for row in r.tables["groups"]]
    if labels_comp is None:
        labels_comp = all_groups
    masks[tag] = np.asarray([1 if g in retained else 0 for g in labels_comp], dtype=np.int8)

multiplicity = analyse_circuit_multiplicity(
    masks,
    component_labels=labels_comp,
    stable_threshold=0.80,
)
print(multiplicity.summary())
multiplicity.figure(mode=PLOT_MODE).show()


## DS14 — causal synergy


In [ ]:
synergy = session.causal_synergy(
    (
        UnitGroup("net.2", tuple(range(0, 8)), label="block_a"),
        UnitGroup("net.2", tuple(range(8, 16)), label="block_b"),
        UnitGroup("net.4", tuple(range(0, 8)), label="block_c"),
    ),
    baseline="zero",
)
print(synergy.summary())
synergy.figure(mode=PLOT_MODE).show()


## DS15 — activation mediation


In [ ]:
mediation = session.activation_mediation(
    torch.zeros_like(x_focus),
    layer_names=LINEAR_LAYERS[:3],
    include_reverse_noising=True,
)
print(mediation.summary())
mediation.figure(mode=PLOT_MODE).show()


## DS16 — topological shift


In [ ]:
topo = analyse_topological_shift(
    dark_dataset,
    dark_dataset_noisy,
    include_h1=False,
    maximum_points=64,
    seed=17,
)
print(topo.summary())
topo.figure(mode=PLOT_MODE).show()


## DS17 — explanation uncertainty


In [ ]:
repeats = []
for seed in range(6):
    g = torch.Generator().manual_seed(17 + seed)
    x_r = x_focus + 0.05 * torch.randn(x_focus.shape, generator=g)
    r = vraven.explain(model, x_r, target=TARGET, mode="gradient")
    vec = np.abs(np.asarray(r.trace.inputs.attribution_values, dtype=np.float64)).reshape(-1)
    repeats.append(vec)

uncertainty = analyse_explanation_uncertainty(np.vstack(repeats), top_k=15)
print(uncertainty.summary())
uncertainty.figure(mode=PLOT_MODE).show()


## DS18 — high-confidence OOD


In [ ]:
fps = np.asarray(dark_dataset.fingerprints, dtype=np.float64)
conf = probs[:, 1]
ood = analyse_high_confidence_ood(
    fps[: max(4, len(fps)//2)],
    fps[max(4, len(fps)//2) :],
    query_confidence=conf[max(4, len(fps)//2) :],
)
print(ood.summary())
ood.figure(mode=PLOT_MODE).show()


## DS19 — internal Linear anatomy (printable insides)

Live `W·a` anatomy for each Linear layer: top input energy-routing units and top output pre-activations. This is measured tensor anatomy, not a decorative network drawing.


In [ ]:
anatomy = inspect_internal_anatomy(
    model,
    x_focus,
    target=TARGET,
    layer_names=LINEAR_LAYERS,
    top_units=10,
)
print(anatomy.summary())
print(print_internal_anatomy(anatomy))
anatomy.figure(mode=PLOT_MODE).show()


## DS20 — measured circuit wiring (nodes + edges)

Takes the ε-sufficient core from DS01 (`circuit`) and draws measured couplings between retained groups:

`mass(G → H) = Σ |W_ji| · |a_i|`

Edge width/opacity follow that mass. This is the wiring of an intervention core, not a claim of unique causal pathways.


In [ ]:
wiring = build_circuit_wiring(model, x_focus, circuit, include_removed=False)
print(wiring.summary())
print("edges:", wiring.metrics["edges"], "total mass:", round(wiring.metrics["total_edge_mass"], 4))
wiring.figure(mode=PLOT_MODE).show()


## DS21 — class-conditional unit field

Dense layer×unit heatmap of Hedges’ *g* between benign and malware cohorts. Complementary to DS03’s layer summary: this shows the internal field, not just the strongest units.


In [ ]:
unit_field = map_class_conditional_unit_field(
    dark_dataset,
    class_a=0,
    class_b=1,
    max_units_per_layer=64,
)
print(unit_field.summary())
unit_field.figure(mode=PLOT_MODE).show()
